In [1]:
import pandas as pd
import os
import glob

# Path to your data folder
data_dir = "GSE260657_RAW/"

# Get all txt.gz files
files = sorted(glob.glob(os.path.join(data_dir, "*.txt.gz")))

dfs = []
for f in files:
    sample_name = os.path.basename(f).replace(".txt.gz", "")
    df = pd.read_csv(f, sep="\t", compression="gzip", index_col=0)
    df.columns = [f"{sample_name}_{col}" if len(df.columns) > 1 else sample_name 
                  for col in df.columns]
    dfs.append(df)

# Combine all samples
combined = pd.concat(dfs, axis=1)
combined = combined.fillna(0)

print(combined.shape)
print(combined.head())

# Save combined matrix
combined.to_csv("GSE260657_combined.csv.gz", compression="gzip")
print("Saved!")

(28657, 7690)
                 GSM8121894_athero_human_1_Plate_730s_B07  \
LOC100996442                                            0   
ENSG00000239945                                         0   
CICP27                                                  0   
ENSG00000268903                                         0   
ENSG00000269981                                         0   

                 GSM8121894_athero_human_1_Plate_730s_B08  \
LOC100996442                                            0   
ENSG00000239945                                         0   
CICP27                                                  1   
ENSG00000268903                                         0   
ENSG00000269981                                         0   

                 GSM8121894_athero_human_1_Plate_730s_B10  \
LOC100996442                                            0   
ENSG00000239945                                         0   
CICP27                                                 30   
ENSG0000

In [7]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import os

print("Loading combined file...")
combined = pd.read_csv("GSE260657_combined.csv.gz", index_col=0, compression="gzip")
print(f"Loaded: {combined.shape}")

Loading combined file...
Loaded: (28657, 7690)


In [8]:
# print(combined.shape)
# print(combined.head())
# print(combined.dtypes.value_counts())

import scanpy as sc
import numpy as np

# Convert combined to AnnData (cells as rows)
adata = sc.AnnData(combined.T)  # shape: (cells × genes)

# Add sample info from column names
adata.obs["sample"] = [col.split("_athero_human_")[0] for col in combined.columns]
adata.obs["human"] = [col.split("_athero_human_")[1].split("_")[0] for col in combined.columns]

print(adata)

AnnData object with n_obs × n_vars = 7690 × 28657
    obs: 'sample', 'human'


In [9]:
import scanpy as sc
import matplotlib.pyplot as plt
import os

os.makedirs("plots", exist_ok=True)

# ── 1. Filter ─────────────────────────────────────────────────────────
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# ── Extra: remove all-zero genes ──────────────────────────────────────
import numpy as np
gene_mask = np.asarray(adata.X.sum(axis=0)).flatten() > 0
adata = adata[:, gene_mask]
print(f"After zero filter: {adata.shape}")

# ── 2. Normalize & Log ────────────────────────────────────────────────
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ── 3. Save raw BEFORE scaling ────────────────────────────────────────
adata.raw = adata

# ── 4. Highly Variable Genes ──────────────────────────────────────────
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
adata = adata[:, adata.var.highly_variable]
print(f"After HVG filter: {adata.shape}")

# ── 5. Scale ──────────────────────────────────────────────────────────
sc.pp.scale(adata, max_value=10)



# ── 6. QC metrics from raw ────────────────────────────────────────────
adata_raw = adata.raw.to_adata()
sc.pp.calculate_qc_metrics(adata_raw, inplace=True)
adata.obs['n_genes_by_counts'] = adata_raw.obs['n_genes_by_counts']
adata.obs['total_counts']      = adata_raw.obs['total_counts']

# ── 7. PCA ────────────────────────────────────────────────────────────
sc.tl.pca(adata, svd_solver='arpack')

sc.pl.pca(adata, color='sample', title='PCA by Sample', show=False)
plt.savefig("plots/PCA_by_sample.png", dpi=150, bbox_inches='tight')
plt.close()

sc.pl.pca(adata, color='human', title='PCA by Donor', show=False)
plt.savefig("plots/PCA_by_donor.png", dpi=150, bbox_inches='tight')
plt.close()

sc.pl.pca_variance_ratio(adata, n_pcs=50, show=False)
plt.savefig("plots/PCA_variance_ratio.png", dpi=150, bbox_inches='tight')
plt.close()

# ── 8. UMAP ───────────────────────────────────────────────────────────
sc.pp.neighbors(adata, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

sc.pl.umap(adata, color='leiden', title='UMAP - Clusters',
           legend_loc='on data', show=False)
plt.savefig("plots/UMAP_clusters.png", dpi=150, bbox_inches='tight')
plt.close()

sc.pl.umap(adata, color='sample', title='UMAP - Sample', show=False)
plt.savefig("plots/UMAP_by_sample.png", dpi=150, bbox_inches='tight')
plt.close()

sc.pl.umap(adata, color='human', title='UMAP - Donor', show=False)
plt.savefig("plots/UMAP_by_donor.png", dpi=150, bbox_inches='tight')
plt.close()

# ── 9. Violin ─────────────────────────────────────────────────────────
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts'],
             groupby='human', jitter=0.4, rotation=45, show=False)
plt.savefig("plots/violin_QC.png", dpi=150, bbox_inches='tight')
plt.close()

print("All plots saved to /plots folder!")

After zero filter: (7690, 28657)


c:\Users\shail\Desktop\Shailja_everything\CMU_courses\coursesenv\Lib\site-packages\scanpy\preprocessing\_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


After HVG filter: (7690, 2000)


c:\Users\shail\Desktop\Shailja_everything\CMU_courses\coursesenv\Lib\site-packages\scanpy\preprocessing\_scale.py:309: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
c:\Users\shail\Desktop\Shailja_everything\CMU_courses\coursesenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\shail\AppData\Local\Temp\ipykernel_37956\3184443806.py:58: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=0.5)


All plots saved to /plots folder!


In [11]:
# ── Save for R ────────────────────────────────────────────────────────
adata_save = adata.raw.to_adata()
adata_save.obs['leiden'] = adata.obs['leiden']
adata_save.obs['sample'] = adata.obs['sample']
adata_save.obs['human']  = adata.obs['human']

# Clear uns to avoid log1p R compatibility issue
adata_save.uns = {}

adata_save.write_h5ad("GSE260657_filtered.h5ad")
print("Saved!")

Saved!
